In [1]:
import os
import pickle
import json
import re
import numpy as np
import pandas as pd
import tqdm

In [2]:
# Function Definitions

def load_json_files(directory_path):
    """
    Load all JSON files from the specified directory and return a dictionary of the data.
    """
    loaded_data = {}
    json_files = [f for f in os.listdir(directory_path) if f.endswith('.json')]
    
    for i, filename in enumerate(json_files, start=1):
        file_path = os.path.join(directory_path, filename)
        with open(file_path, 'r') as file:
            loaded_data[filename] = json.load(file)
        print(f"Loaded {i}/{len(json_files)} files: {filename}")
    
    return loaded_data

def extract_all_contexts(text, query_words, window_size=50):
    """
    Extract all context windows around each occurrence of the query words in the text.
    Returns a list of contexts, or an empty list if no occurrences are found.
    The search is case-insensitive.
    """
    pattern = r'\b(?:' + '|'.join(re.escape(word) for word in query_words) + r')\b'
    matches = [match for match in re.finditer(pattern, text, re.IGNORECASE)]
    
    contexts = []
    for match in matches:
        start = max(match.start() - window_size, 0)
        end = min(match.end() + window_size, len(text))
        contexts.append(text[start:end])
    
    return contexts

def normalize_roman(roman):
    """
    Normalize lowercase 'l' to uppercase 'I' and handle common typos such as 'lV' to 'IV'.
    """
    normalization_map = {
        'lll': 'III',
        'll': 'II',
        'lV': 'IV',
        'l': 'I'
    }
    return normalization_map.get(roman, roman.upper())  # Return the normalized form if it exists, otherwise convert to uppercase

def roman_to_int(roman):
    """
    Convert a Roman numeral to an integer.
    """
    roman = normalize_roman(roman)
    
    roman_numerals = {
        'I': 1, 'V': 5, 'X': 10,
        'L': 50, 'C': 100, 'D': 500, 'M': 1000  # M, D, C are not used for staging annotations
    }
    int_val = 0
    for i in range(len(roman)):
        if i > 0 and roman_numerals[roman[i]] > roman_numerals[roman[i - 1]]:
            int_val += roman_numerals[roman[i]] - 2 * roman_numerals[roman[i - 1]]
        else:
            int_val += roman_numerals[roman[i]]
    return int_val

def interpret_special_cases(stage_num):
    """
    Interpret special cases such as '111' and '11' as stages '3' and '2', respectively.
    """
    special_cases = {'111': '3', '11': '2'}
    return special_cases.get(stage_num, stage_num)

def extract_cancer_stages(context):
    """
    Extract all cancer stages from a given context string.
    Returns a concatenated string of stages found in the text, separated by ', '.
    Converts stages like IIIA to 3a, etc.
    Handles special cases like '111' as stage 3 and '11' as stage 2.
    """
    pattern = r'\b(?:Stage|Staging)\s*([IVXL]+|l{1,3}|lV)([A-Da-d]?)\b|\b(?:Stage|Staging)\s*(\d+)([A-Da-d]?)\b'
    matches = re.findall(pattern, context, re.IGNORECASE)
    
    stages = []
    for match in matches:
        stage_num = match[0] if match[0] else match[2]  # Roman numeral or Arabic number
        stage_suffix = match[1].upper() if match[1] else (match[3].upper() if match[3] else '')  # Suffix, if present
        
        stage_num = interpret_special_cases(stage_num)
        
        if stage_num.isdigit():
            stage = stage_num
        else:
            stage = str(roman_to_int(stage_num))
        
        if stage_suffix:
            stage += stage_suffix.lower()
        
        stages.append(stage)
    
    return ', '.join(stages)

def merge_stages(stages):
    """
    Merge extracted cancer stages into a single value.
    If all stages are the same, return that value.
    Exclude stages outside the range 1-4. If there are more than 3 unique valid stages, return None.
    """
    stage_list = [stage.strip() for stage in stages.split(', ') if stage.strip()]  # Clean up and filter out empty strings
    
    # Convert stages to integers, stripping any suffixes, and filter those within the range 1-4
    numeric_stages = []
    for stage in stage_list:
        if stage[-1].isalpha():
            numeric_value = int(stage[:-1])
        else:
            numeric_value = int(stage)
        
        if 1 <= numeric_value <= 4:
            numeric_stages.append(numeric_value)
    
    unique_stages = set(numeric_stages)
    
    # If there are no valid stages, return None
    if not unique_stages:
        return None
    
    # If more than 3 unique stages, return None
    if len(unique_stages) > 3:
        return None
    
    # If all stages are the same, return that value; otherwise, return the highest valid stage
    return unique_stages.pop() if len(unique_stages) == 1 else max(unique_stages)


In [3]:
# Main Script

# Load JSON files from the specified directory
directory_path = '../../data/gusev/PROFILE/CLINICAL/OncDRS/CLINICAL_TEXTS_2024_03'
loaded_data = load_json_files(directory_path)

Loaded 1/28 files: RequestID-19-033-March24-Imaging-1.json
Loaded 2/28 files: RequestID-19-033-March24-Imaging-2.json
Loaded 3/28 files: RequestID-19-033-March24-Imaging-3.json
Loaded 4/28 files: RequestID-19-033-March24-Imaging-4.json
Loaded 5/28 files: RequestID-19-033-March24-Imaging-5.json
Loaded 6/28 files: RequestID-19-033-March24-Imaging-6.json
Loaded 7/28 files: RequestID-19-033-March24-Imaging-7.json
Loaded 8/28 files: RequestID-19-033-March24-Path-1.json
Loaded 9/28 files: RequestID-19-033-March24-Path-10.json
Loaded 10/28 files: RequestID-19-033-March24-Path-2.json
Loaded 11/28 files: RequestID-19-033-March24-Path-3.json
Loaded 12/28 files: RequestID-19-033-March24-Path-4.json
Loaded 13/28 files: RequestID-19-033-March24-Path-5.json
Loaded 14/28 files: RequestID-19-033-March24-Path-6.json
Loaded 15/28 files: RequestID-19-033-March24-Path-7.json
Loaded 16/28 files: RequestID-19-033-March24-Path-8.json
Loaded 17/28 files: RequestID-19-033-March24-Path-9.json
Loaded 18/28 files

In [17]:
# Initialize a list to store the data for the DataFrame
records = []
query_words = ["stage"]
window_size = 200

In [18]:
# Process the loaded data to extract contexts
for i, (file_name, file_data) in enumerate(loaded_data.items()):
    # Determine RPT_TYPE based on the filename
    if "Imaging" in file_name:
        rpt_type = "ImagingNote"
    elif "Path" in file_name:
        rpt_type = "PathologyNote"
    elif "ProgNote" in file_name:
        rpt_type = "ProgressNote"
    else:
        rpt_type = "Unknown"  # In case none of the expected patterns match

    for doc in file_data['response']['docs']:
        dfci_mrn = doc.get('DFCI_MRN', None)
        event_date = doc.get('EVENT_DATE', None)
        rpt_date = doc.get('RPT_DATE', None)
        rpt_text = doc.get('RPT_TEXT', "")
        dept_id = doc.get('DEPT_ID', None)
        provider_department = doc.get('PROVIDER_DEPARTMENT', None)
        
        # Extract contexts around the query words
        contexts = extract_all_contexts(rpt_text, query_words, window_size=window_size)
        
        if contexts:
            concatenated_context = '; '.join(contexts)
            records.append({
                'DFCI_MRN': dfci_mrn,
                'EVENT_DATE': event_date,
                'RPT_DATE': rpt_date,
                'RPT_TEXT': concatenated_context,
                'RPT_TEXT_WHOLE': rpt_text,
                'RPT_LENGTH': len(rpt_text),
                'DEPT_ID': dept_id,
                'PROVIDER_DEPARTMENT': provider_department,
                'RPT_TYPE': rpt_type  # Include the RPT_TYPE in the record
            })
            
            
    print(f"Processed {i+1}/{len(loaded_data)} files: {file_name}")

# Create DataFrame from the records
df = pd.DataFrame(records)

# Display the DataFrame
# print(df.head())

Processed 1/28 files: RequestID-19-033-March24-Imaging-1.json
Processed 2/28 files: RequestID-19-033-March24-Imaging-2.json
Processed 3/28 files: RequestID-19-033-March24-Imaging-3.json
Processed 4/28 files: RequestID-19-033-March24-Imaging-4.json
Processed 5/28 files: RequestID-19-033-March24-Imaging-5.json
Processed 6/28 files: RequestID-19-033-March24-Imaging-6.json
Processed 7/28 files: RequestID-19-033-March24-Imaging-7.json
Processed 8/28 files: RequestID-19-033-March24-Path-1.json
Processed 9/28 files: RequestID-19-033-March24-Path-10.json
Processed 10/28 files: RequestID-19-033-March24-Path-2.json
Processed 11/28 files: RequestID-19-033-March24-Path-3.json
Processed 12/28 files: RequestID-19-033-March24-Path-4.json
Processed 13/28 files: RequestID-19-033-March24-Path-5.json
Processed 14/28 files: RequestID-19-033-March24-Path-6.json
Processed 15/28 files: RequestID-19-033-March24-Path-7.json
Processed 16/28 files: RequestID-19-033-March24-Path-8.json
Processed 17/28 files: Requ

In [19]:
# Extract and merge stages
stages = []
stages_merged = []
for idx, entry in tqdm.tqdm(df.iterrows(), total=len(df)):
    report_txt = entry['RPT_TEXT']
    stage_raw = extract_cancer_stages(report_txt)
    stage_merged = merge_stages(stage_raw) if stage_raw else None
    
    stages.append(stage_raw)
    stages_merged.append(stage_merged)
    
df['DERIVED_STAGE_RAW'] = stages
df['DERIVED_STAGE_MERGED'] = stages_merged

100%|██████████| 939376/939376 [01:40<00:00, 9330.66it/s] 


In [20]:
df = df.sort_values(by=['DFCI_MRN', 'EVENT_DATE'])
df.index = np.arange(len(df))
df.to_csv('dfci_cancer_stage_notes.csv')

In [21]:
df

,DFCI_MRN,EVENT_DATE,RPT_DATE,RPT_TEXT,RPT_TEXT_WHOLE,RPT_LENGTH,DEPT_ID,PROVIDER_DEPARTMENT,RPT_TYPE,DERIVED_STAGE_RAW,DERIVED_STAGE_MERGED
0,1000012,2021-05-29T14:30:00Z,2021-06-15T15:41:00Z,cess and distal appendiceal luminal obliterati...,CASE: BS-21-R33268\r\nPatient Name: CAROL LAFL...,3998,None,None,PathologyNote,,NaN
1,1000013,2021-05-14T19:08:00Z,2021-05-19T09:45:00Z,ous metaplasia.\r\n Sessile serrated lesio...,CASE: BS-21-D29623\r\nPATIENT: COLLEEN ROOHAN\...,3606,None,None,PathologyNote,,NaN
2,1000013,2021-06-08T20:00:00Z,None,tube.\n\n\nA/P: Colleen Roohan is a 23 y.o. f...,\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n...,6512,10030010115,BWP GENERAL GI SURG,ProgressNote,,NaN
3,1000013,2021-07-13T21:00:00Z,None,undergo a right hemicolectomy and lymph node s...,\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n...,5345,10030010115,BWP GENERAL GI SURG,ProgressNote,,NaN
4,1000013,2021-07-14T15:30:00Z,None,and osseous metaplasia.\n\n\n\nSessile serrate...,HISTORY OF PRESENT ILLNESS: Oncology History ...,10900,14010090011,DF GI ONC CH,ProgressNote,,NaN
...,...,...,...,...,...,...,...,...,...,...,...
939371,999995,2022-05-18T21:20:00Z,None,ronic disease Vitamin B12 deficiency Hyperte...,This is a 74 y.o. male who presents for follo...,5355,14010080001,DF MEDICAL ONC MV,ProgressNote,2,2.0
939372,999995,2023-05-22T21:20:00Z,None,ronic disease Vitamin B12 deficiency Hyperte...,This is a 75 y.o. male who presents for follo...,6375,14010080001,DF MEDICAL ONC MV,ProgressNote,2,2.0
939373,999995,2023-07-17T22:20:00Z,None,ronic disease Vitamin B12 deficiency Hyperte...,This is a 75 y.o. male who presents for f/u a...,5686,14010080001,DF MEDICAL ONC MV,ProgressNote,2,2.0
939374,999995,2023-09-11T21:40:00Z,None,ronic disease Vitamin B12 deficiency Hyperte...,This is a 75 y.o. male who presents for f/u a...,5779,14010080001,DF MEDICAL ONC MV,ProgressNote,2,2.0


In [22]:
# Derived from mrns_extractor_based_on_genomics.ipynb
# Loading the list from local storage
with open('lung_cancer_mrns.pkl', 'rb') as file:
    lung_mrns_loaded = pickle.load(file)
lung_mrns_loaded = [str(mrn) for mrn in lung_mrns_loaded]
df['DFCI_MRN'] = df['DFCI_MRN'].astype(str)

In [23]:
df_lung = df.loc[df.DFCI_MRN.isin(lung_mrns_loaded)]
df_sorted = df_lung.sort_values(by=['DFCI_MRN', 'EVENT_DATE'])
df_sorted.index = np.arange(len(df_sorted))
df_sorted.to_csv('dfci_lung_cancer_stage_notes.csv', index=False)

In [24]:
df_sorted

,DFCI_MRN,EVENT_DATE,RPT_DATE,RPT_TEXT,RPT_TEXT_WHOLE,RPT_LENGTH,DEPT_ID,PROVIDER_DEPARTMENT,RPT_TYPE,DERIVED_STAGE_RAW,DERIVED_STAGE_MERGED
0,1000172,2021-05-25T19:30:00Z,None,PATIENT ID\n\nMr. Lovetere is a 60 y.o. male w...,PATIENT ID\n\nMr. Lovetere is a 60 y.o. male w...,7561,14010090031,DF THORACIC ONC CH,ProgressNote,4,4.0
1,1000172,2021-06-14T23:15:00Z,None,delightful 60 y.o.-year-old male who presents...,DERMATOLOGY CLINIC PATIENT NOTE Brigham and...,5517,10030040059,BWP DERM FXB,ProgressNote,4,4.0
2,1000172,2021-06-15T19:00:00Z,None,PATIENT ID\n\nMr. Lovetere is a 60 y.o. male w...,PATIENT ID\n\nMr. Lovetere is a 60 y.o. male w...,8157,14010090031,DF THORACIC ONC CH,ProgressNote,4,4.0
3,1000172,2021-06-30T18:30:00Z,None,06/30/21\n\nSubjective:\n\n Patient ID: Peter ...,06/30/21\n\nSubjective:\n\n Patient ID: Peter ...,7003,14010010314,DF THORACIC ONC,ProgressNote,"4, 4",4.0
4,1000172,2021-07-14T17:00:00Z,None,07/14/21\n\nSubjective:\n\n Patient ID: Peter ...,07/14/21\n\nSubjective:\n\n Patient ID: Peter ...,7777,14010010314,DF THORACIC ONC,ProgressNote,"4, 4",4.0
...,...,...,...,...,...,...,...,...,...,...,...
99755,999964,2021-06-29T21:30:00Z,None,\n\nThoracic Medical Oncology Follow up Visit\...,\n\nThoracic Medical Oncology Follow up Visit\...,13225,14010010314,DF THORACIC ONC,ProgressNote,"4, 4, 4, 4",4.0
99756,999964,2021-07-23T23:00:00Z,None,\n\nThoracic Medical Oncology Follow up Visit\...,\n\nThoracic Medical Oncology Follow up Visit\...,13941,14010090031,DF THORACIC ONC CH,ProgressNote,"4, 4",4.0
99757,999964,2021-08-03T17:30:00Z,None,ation - Initial Visit\n\n ? Diagnosis and tr...,DANA FARBER CANCER INSTITUTE Nutrition Consul...,5080,14010090023,DF NUTRITION CH,ProgressNote,4,4.0
99758,999964,2021-08-05T11:10:00Z,2021-08-05T21:39:00Z,o.: 22D0666524\r\nLaboratory Director: Stacy E...,CASE: BL-21-F35364\r\nPATIENT: MAGNOLIA CUERVO...,1508,10030010390,BWH IMG IR CSIR,PathologyNote,4,4.0


In [28]:
print(df_sorted.RPT_TEXT_WHOLE.values[0])

PATIENT ID

Mr. Lovetere is a 60 y.o. male with recently diagnosed, stage IV non-smalll cell lung cancer, adenocarcinoma histology.  He is referred to clinic to discuss next steps in management.

ONCOLOGY HISTORY  Oncology History Overview Note
5/3/2021: Presented to MGH ER after experiencing hemoptysis after sneezing.

 5/3/2021: CXR: Round opacity projecting over the right lung, possibly with central cavitation

5/3/2021: CT chest: Right upper lobe mass measuring 4.1 x 2.8 cm. Ground glass opacities in the right middle lobe and left upper lobe likely reflect infectious or inflammatory process. Multiple hypodense hepatic lesions and mediastinal/right hilar lymphadenopathy.

 5/3/2021: Discharged home with plan for f/u with PCP to arrange for further
workup.

  5/6/2021: Thoracic surgery consult Matthew Rochefort, MD  -recommended bronchoscopy EBUS  -If tumor is confined to RUL then recommend right video assisted upper lobectomy

5/6/2021: PFT  FEV1  pre  3.34  97%  FVC  pre  4.75  106

#### Perform additional filtering

In [25]:
df_filtered = df.dropna(subset=['DERIVED_STAGE_MERGED'])

In [26]:
df_lung = df_filtered.loc[df_filtered.DFCI_MRN.isin(lung_mrns_loaded)]
df_sorted = df_lung.sort_values(by=['DFCI_MRN', 'EVENT_DATE'])
df_sorted.index = np.arange(len(df_sorted))
df_sorted.to_csv('dfci_lung_cancer_stage_notes_filtered.csv', index=False)

In [27]:
# Convert the 'EVENT_DATE' column to datetime
df_filtered['EVENT_DATE'] = pd.to_datetime(df_filtered['EVENT_DATE'])

# Sort df_filtered first by DFCI_MRN and then by EVENT_DATE
df_filtered_sorted = df_filtered.sort_values(by=['DFCI_MRN', 'EVENT_DATE'])
df_filtered_sorted.index = np.arange(len(df_filtered_sorted))
df_filtered_sorted.to_csv('dfci_cancer_stage_notes_filtered.csv', index=False)

/opt/anaconda/4.6.14/lib/python3.6/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [ ]:
## run until here!!

### Use cases:

In [21]:
# Example use cases:

# Case 1: Stage IIIc and Stage IV
context1 = "Additional history was obtained via the electronic medical record: Stage IIIc adenocarcinoma of the lung. Stage IV, Stage IIId."
stages1 = extract_cancer_stages(context1)
merged_stage1 = merge_stages(stages1)
print(f"Context: {context1}")
print(f"Extracted Cancer Stages: {stages1}")
print(f"Merged Cancer Stage: {merged_stage1}")
print()

# Case 2: Special cases 111 and 11
context2 = "Patient diagnosed with Stage 111 cancer with some metastasis, followed by an earlier Stage 11 diagnosis."
stages2 = extract_cancer_stages(context2)
merged_stage2 = merge_stages(stages2)
print(f"Context: {context2}")
print(f"Extracted Cancer Stages: {stages2}")
print(f"Merged Cancer Stage: {merged_stage2}")
print()

# Case 3: Stage IIA and IIIb
context3 = "Stage IIA colon cancer detected. Staging report also noted Stage IIIb."
stages3 = extract_cancer_stages(context3)
merged_stage3 = merge_stages(stages3)
print(f"Context: {context3}")
print(f"Extracted Cancer Stages: {stages3}")
print(f"Merged Cancer Stage: {merged_stage3}")
print()

# Case 4: Multiple stage mentions including special cases
context4 = "Stage 111 are all present in the patient records. This patient has a history of stage 1 breast cancer"
stages4 = extract_cancer_stages(context4)
merged_stage4 = merge_stages(stages4)
print(f"Context: {context4}")
print(f"Extracted Cancer Stages: {stages4}")
print(f"Merged Cancer Stage: {merged_stage4}")
print()

# Case 5: "stageiii", "stage3", "stageiiia"
context1 = "The patient has been diagnosed with stageiii, stage3, and stageiiia cancer."
stages1 = extract_cancer_stages(context1)
merged_stage1 = merge_stages(stages1)
print(f"Context: {context1}")
print(f"Extracted Cancer Stages: {stages1}")
print(f"Merged Cancer Stage: {merged_stage1}")
print()

# Case 6: "stageIV", "stageiiib", "stage2"
context2 = "The patient was found to have stage IV melanoma, with previous stageiiib and stage2 diagnoses."
stages2 = extract_cancer_stages(context2)
merged_stage2 = merge_stages(stages2)
print(f"Context: {context2}")
print(f"Extracted Cancer Stages: {stages2}")
print(f"Merged Cancer Stage: {merged_stage2}")
print()

# Case 7: Handling valid stages and excluding invalid ones
context1 = "The patient has been diagnosed with stage lV, with a history of stage ll and stage l diagnoses"
stages1 = extract_cancer_stages(context1)
merged_stage1 = merge_stages(stages1)
print(f"Context: {context1}")
print(f"Extracted Cancer Stages: {stages1}")
print(f"Merged Cancer Stage: {merged_stage1}")
print()

# Case 8: Handling mixed stages and returning the highest valid stage
context2 = "The patient was diagnosed with stage IV, stage iiiA, and stage ll."
stages2 = extract_cancer_stages(context2)
merged_stage2 = merge_stages(stages2)
print(f"Context: {context2}")
print(f"Extracted Cancer Stages: {stages2}")
print(f"Merged Cancer Stage: {merged_stage2}")
print()

# Case 9: Handling more than 3 unique valid stages
context3 = "stage chart: stage 1, stage 2, stage 3, and stage 4."
stages3 = extract_cancer_stages(context3)
merged_stage3 = merge_stages(stages3)
print(f"Context: {context3}")
print(f"Extracted Cancer Stages: {stages3}")
print(f"Merged Cancer Stage: {merged_stage3}")
print()

# Case 10: Handling stages outside the range 1-4
context4 = "The patient has stage 2, with a history of stage 1 dianosis. stage 5 -- typo"
stages4 = extract_cancer_stages(context4)
merged_stage4 = merge_stages(stages4)
print(f"Context: {context4}")
print(f"Extracted Cancer Stages: {stages4}")
print(f"Merged Cancer Stage: {merged_stage4}")
print()

Context: Additional history was obtained via the electronic medical record: Stage IIIc adenocarcinoma of the lung. Stage IV, Stage IIId.
Extracted Cancer Stages: 3c, 4, 3d
Merged Cancer Stage: 4

Context: Patient diagnosed with Stage 111 cancer with some metastasis, followed by an earlier Stage 11 diagnosis.
Extracted Cancer Stages: 3, 2
Merged Cancer Stage: 3

Context: Stage IIA colon cancer detected. Staging report also noted Stage IIIb.
Extracted Cancer Stages: 2a, 3b
Merged Cancer Stage: 3

Context: Stage 111 are all present in the patient records. This patient has a history of stage 1 breast cancer
Extracted Cancer Stages: 3, 1
Merged Cancer Stage: 3

Context: The patient has been diagnosed with stageiii, stage3, and stageiiia cancer.
Extracted Cancer Stages: 3, 3, 3a
Merged Cancer Stage: 3

Context: The patient was found to have stage IV melanoma, with previous stageiiib and stage2 diagnoses.
Extracted Cancer Stages: 4, 3b, 2
Merged Cancer Stage: 4

Context: The patient has been 

In [ ]:
df_filtered

In [ ]:
# With DFCI GPT4 API vailable 
# We can validate the staging results;